In [5]:
import pandas as pd

def load_and_clean(filepath, daily_minutes=1380):
    df = pd.read_csv(filepath, header=None,
                     names=["datetime", "open", "high", "low", "close", "volume"])
    
    df["contract"] = filepath.split("/")[-1].replace(".csv", "")
    df["datetime"] = pd.to_datetime(df["datetime"], format="%Y.%m.%d.%H:%M:%S")
    df["date"] = df["datetime"].dt.date
    df["bars_per_day"] = df.groupby("date")["datetime"].transform("count")
    
    threshold = int(daily_minutes * 0.90)
    df_clean = df[df["bars_per_day"] >= threshold].copy()
    
    valid_ohlc = (
        (df_clean["high"] >= df_clean["low"]) &
        (df_clean["open"] >= df_clean["low"]) &
        (df_clean["open"] <= df_clean["high"]) &
        (df_clean["close"] >= df_clean["low"]) &
        (df_clean["close"] <= df_clean["high"]) &
        (df_clean["volume"] > 0)
    )
    df_clean = df_clean[valid_ohlc].copy()
    return df_clean

def resample_ohlcv(df, tau):
    
    # Make a copy so original dataframe is untouched
    temp = df.copy()

    # Set datetime as index
    temp = temp.set_index("datetime")

    # Resample OHLCV
    df_resampled = temp.resample(f"{tau}min").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum"
    })

    # Drop incomplete/empty intervals
    df_resampled = df_resampled.dropna()

    # Bring datetime back as a column
    df_resampled = df_resampled.reset_index()

    return df_resampled

def compute_ranges(df, tick_size):
    df = df.copy()
    df["range_ticks"] = ((df["high"] - df["low"]) / tick_size).round().astype(int)
    df["range_up"] = ((df["high"] - df["open"]) / tick_size).round().astype(int)
    df["range_dn"] = ((df["open"] - df["low"]) / tick_size).round().astype(int)
    return df

def compute_epdfs(df):
    epdfs = {}
    for col in ["range_ticks", "range_up", "range_dn"]:
        counts = df[col].value_counts()
        epdfs[col] = (counts / len(df)).sort_index()
    return epdfs

import numpy as np

def compute_ewma(series, m):
    values = series.astype(float).to_numpy()
    
    lambda_ = 2 ** (-1 / m)

    ewma = np.zeros(len(values))
    ewmv = np.zeros(len(values))

    sumW = 0.0
    sumWX = 0.0
    sumWSS = 0.0

    # First row has no previous observation to use
    ewma[0] = np.nan
    ewmv[0] = np.nan

    for j in range(1, len(values)):
        # Use eta_{j-1}, not eta_j
        prev_x = values[j - 1]

        sumW = lambda_ * sumW + 1
        sumWX = lambda_ * sumWX + prev_x

        ewma[j] = sumWX / sumW

        sumWSS = lambda_ * sumWSS + (prev_x - ewma[j]) ** 2
        ewmv[j] = np.sqrt(sumWSS / sumW)

    return pd.DataFrame({
        "ewma": ewma,
        "ewmv": ewmv
    }, index=series.index)

def compute_conditional_epdfs(df, min_count=30):
    cond_epdfs = {}
    grouped = df.groupby(["vol_state", "range_state", "trend_state"])
    
    for state, group in grouped:
        if len(group) < min_count:
            continue
        cond_epdfs[state] = {
            "range_dn": (group["range_dn"].value_counts() / len(group)).sort_index(),
            "range_up": (group["range_up"].value_counts() / len(group)).sort_index()
        }
    
    print(f"Valid state combinations: {len(cond_epdfs)} / 27")
    return cond_epdfs


def get_order_recommendation(signal_direction, open_price, state, 
                              cond_epdfs, tick_size=0.10, target_prob=0.70):
    
    # Pick correct ePDF based on signal direction
    if signal_direction == "buy":
        epdf = cond_epdfs[state].get("range_dn") if isinstance(cond_epdfs[state], dict) else cond_epdfs[state]
        direction = -1  # place below open
    else:
        epdf = cond_epdfs[state].get("range_up") if isinstance(cond_epdfs[state], dict) else cond_epdfs[state]
        direction = 1   # place above open
    
    if epdf is None:
        return None
    
    # Compute survival function P(range >= k)
    cumulative = epdf.sort_index().cumsum()
    survival = 1 - cumulative
    
    # Find k where survival probability >= target
    valid = survival[survival >= target_prob]
    if len(valid) == 0:
        k = 0
    else:
        k = valid.index[-1]
    
    limit_price = open_price + direction * k * tick_size
    
    return {
        "limit_price": round(limit_price, 2),
        "ticks_away": k,
        "fill_probability": round(survival.get(k, 0), 3)
    }

from datetime import datetime, timedelta

def load_signals(filepath):
    df = pd.read_csv(filepath, header=None,
                     names=["date_serial", "hour", "minute", "price", "signal"])
    
    excel_epoch = datetime(1899, 12, 30)
    df["datetime"] = df.apply(
        lambda r: excel_epoch
                  + timedelta(days=int(r["date_serial"]))
                  + timedelta(hours=int(r["hour"]))
                  + timedelta(minutes=int(r["minute"])),
        axis=1
    )
    
    df["signal_direction"] = df["signal"].apply(
        lambda s: "buy" if s > 0 else ("sell" if s < 0 else "flat")
    )
    
    return df[["datetime", "price", "signal", "signal_direction"]].sort_values("datetime").reset_index(drop=True)


In [6]:
import glob
files = sorted(glob.glob("data/EuroStoxx/VG*.csv"))
print(files)

['data/EuroStoxx/VGH22.csv', 'data/EuroStoxx/VGM22.csv']


In [8]:
TICK_SIZE = 0.5
MARKET = "EuroStoxx"

files = ['data/EuroStoxx/VGH22.csv', 'data/EuroStoxx/VGM22.csv']
all_dfs = [load_and_clean(f, daily_minutes=840) for f in files]
df_all = pd.concat(all_dfs).sort_values("datetime").reset_index(drop=True)
print(df_all["contract"].value_counts())
print("Total liquid days:", df_all["date"].nunique())

contract
VGH22    78050
VGM22    76723
Name: count, dtype: int64
Total liquid days: 130


In [9]:
daily_vol = df_all.groupby(["date", "contract"])["volume"].sum().reset_index()

best_contract = daily_vol.loc[
    daily_vol.groupby("date")["volume"].idxmax(),
    ["date", "contract"]
]

df_stitched = df_all.merge(best_contract, on=["date", "contract"]).reset_index(drop=True)

print(df_stitched["contract"].value_counts())
print("Total days after stitching:", df_stitched["date"].nunique())
print(df_stitched.groupby("contract")["date"].agg(["min", "max"]))

contract
VGH22    77014
VGM22    71538
Name: count, dtype: int64
Total days after stitching: 130
                 min         max
contract                        
VGH22     2021-12-13  2022-03-16
VGM22     2022-03-17  2022-06-16


In [10]:
# Cleanup
df_stitched = df_stitched.drop(columns=["bars_per_day"])
df_stitched = df_stitched.reset_index(drop=True)

# Resample and compute ranges
df_5min = compute_ranges(resample_ohlcv(df_stitched, 5), TICK_SIZE)

# EWMA states
volume_state = compute_ewma(df_5min["volume"], m=10)
range_state = compute_ewma(df_5min["range_ticks"], m=10)
df_5min[["ewma_volume", "ewmv_volume"]] = volume_state
df_5min[["ewma_range", "ewmv_range"]] = range_state

# Bin into states
df_5min["vol_state"] = pd.qcut(df_5min["ewma_volume"], q=3, labels=[1, 2, 3])
df_5min["range_state"] = pd.qcut(df_5min["ewma_range"], q=3, labels=[1, 2, 3])
df_5min["delta_x"] = df_5min["open"].diff()
df_5min["trend_state"] = pd.qcut(df_5min["delta_x"], q=3, labels=[1, 2, 3])

# Conditional ePDFs
cond_epdfs = compute_conditional_epdfs(df_5min)

print(df_5min.shape)

Valid state combinations: 26 / 27
(30513, 17)


In [11]:
# Load signals
signals = load_signals("data/EuroStoxx/AIAgent_EuroStoxx.csv")
active_signals = signals[signals["signal_direction"] != "flat"].copy()

# Merge with market state
df_signals = active_signals.merge(
    df_5min[["datetime", "vol_state", "range_state", "trend_state", "open"]],
    on="datetime",
    how="inner"
)

print(f"Signals before merge: {len(active_signals)}")
print(f"Signals after merge: {len(df_signals)}")

# Generate recommendations
df_signals["limit_price"] = None
df_signals["ticks_away"] = None
df_signals["fill_probability"] = None

for idx, row in df_signals.iterrows():
    state = (row["vol_state"], row["range_state"], row["trend_state"])
    if state not in cond_epdfs:
        continue
    result = get_order_recommendation(
        row["signal_direction"],
        row["open"],
        state,
        cond_epdfs,
        tick_size=TICK_SIZE
    )
    if result:
        df_signals.at[idx, "limit_price"] = result["limit_price"]
        df_signals.at[idx, "ticks_away"] = result["ticks_away"]
        df_signals.at[idx, "fill_probability"] = result["fill_probability"]

print("Signals with recommendations:", df_signals["limit_price"].notna().sum())


Signals before merge: 25245
Signals after merge: 21655
Signals with recommendations: 21643


In [12]:
# Add metadata
df_signals["tau"] = 5
df_signals["tick_size"] = TICK_SIZE

# Carry contract through
contract_map = df_stitched[["datetime", "contract"]].drop_duplicates(subset="datetime")
df_signals = df_signals.merge(contract_map, on="datetime", how="left")

# State labels
vol_map = {1: "low", 2: "medium", 3: "high"}
range_map = {1: "low", 2: "medium", 3: "high"}
trend_map = {1: "down", 2: "flat", 3: "up"}

df_signals["vol_state_label"] = df_signals["vol_state"].map(vol_map)
df_signals["range_state_label"] = df_signals["range_state"].map(range_map)
df_signals["trend_state_label"] = df_signals["trend_state"].map(trend_map)

# Export
df_signals.to_csv("data/EuroStoxx/ES_signals_with_recommendations.csv", index=False)
print("Saved successfully")
print(df_signals.shape)

Saved successfully
(21655, 17)
